# 20 - Getting Data In, Getting Models Out

**Section:** More on Data | **Prereqs:** `Cross Validation/DataLoader.ipynb` | **Next:** `More on Data/DepthVsBreadth.ipynb`

Reference notebook: the four ways data arrives, and the one right way to save a
model.

**Data in:** a built-in `torchvision` dataset, a URL read straight into pandas,
a browser upload (Colab), or a mounted Google Drive.

**Model out:** save `model.state_dict()` - a plain dict of parameter tensors -
not the model object. Pickling the object hardcodes your class definitions and
file layout, so it breaks as soon as you refactor. A `state_dict` needs the class
to exist to load, but it is portable across code changes and across machines.

```python
torch.save(model.state_dict(), 'model.pt')       # save
model = MyModel()                                 # rebuild the architecture
model.load_state_dict(torch.load('model.pt'))     # fill in the weights
model.eval()                                      # inference mode
```

**Two things this notebook does not show but you should do:** call `model.eval()`
after loading (otherwise dropout and batch-norm stay in training mode), and pass
`map_location='cpu'` when loading GPU-trained weights on a CPU machine.

#Import from torchvision

In [1]:
# torchvision downloads and caches CIFAR-10 (60000 32x32 colour images,
# 10 classes). Same API as MNIST - root, train, download, transform.
import torchvision

# download the CIFAR10 dataset
cdata = torchvision.datasets.CIFAR10(root='cifar10', download=True)

print(cdata)


100%|██████████| 170M/170M [00:02<00:00, 76.1MB/s]


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: cifar10
    Split: Train


In [2]:
# Datasets that come with torchvision: https://pytorch.org/vision/stable/index.html

#Download from the web

In [4]:
# pandas reads a remote spreadsheet directly from its URL.
# header=5 skips five rows of title text before the real column names.
import pandas as pd

# url
marriage_url = 'https://www.cdc.gov/nchs/data/dvs/state-marriage-rates-90-95-99-19.xlsx'

# import directly into pandas
data = pd.read_excel(marriage_url,header=5)
data.head(10)


,Unnamed: 0,2019,2018,2017,2016,2015,2014,2013,2012,2011,...,2006,2005,2004,2003,2002,2001,2000,1999,1995,1990
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alabama,6.697687,6.760408,7.047340,7.147821,7.351544,7.806776,7.817785,8.2,8.4,...,9.2,9.2,9.4,9.6,9.9,9.4,10.1,10.8,9.8,10.6
2,Alaska,6.512245,6.683952,6.914078,7.103441,7.407588,7.508836,7.293928,7.2,7.8,...,8.2,8.2,8.5,8.1,8.3,8.1,8.9,8.6,9.0,10.2
3,Arizona,5.302995,5.534434,5.834867,5.930541,5.922469,5.780449,5.401091,5.6,5.7,...,6.5,6.6,6.7,6.5,6.7,7.6,7.5,8.2,8.8,10.0
4,Arkansas,8.377284,8.863156,9.456845,9.860962,10.040279,10.112026,9.751052,10.9,10.4,...,12.4,12.9,13.4,13.4,14.3,14.3,15.4,14.8,14.4,15.3
5,California 1,5.723191,6.035132,6.278250,6.463590,6.184957,6.441492,6.460467,6.0,5.8,...,6.3,6.4,6.4,6.1,6.2,6.5,5.8,6.4,6.3,7.9
6,Colorado,7.273297,7.585728,7.333845,7.425443,6.791807,7.061603,6.452664,6.8,7.0,...,7.2,7.6,7.4,7.8,8,8.2,8.3,8.2,9.0,9.8
7,Connecticut,5.048401,5.278133,5.553784,5.617858,5.292009,5.368845,5.021023,5.2,5.5,...,5.5,5.8,5.8,5.5,5.7,5.4,5.7,5.8,6.6,7.9
8,Delaware,4.951919,5.237957,5.528417,5.613062,5.712872,6.022783,6.571976,5.8,5.2,...,5.9,5.9,6.1,6,6.4,6.5,6.5,6.7,7.3,8.4
9,District of Columbia,7.773302,7.835377,8.239526,8.149214,8.220425,11.821343,10.791261,8.4,8.7,...,4,4.1,5.2,5.1,5.1,6.2,4.9,6.6,6.1,8.2


#Upload from hard drive

In [5]:
# Colab-only: opens a browser file picker.
from google.colab import files
uploaded = files.upload()


#Map your google-drive

In [6]:
# Colab-only: mounts Google Drive at /content/gdrive so it behaves like a local
# folder. The usual way to persist data and checkpoints across Colab sessions.
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


# Saving the model

In [21]:
# A factory function for the architecture. This matters: loading a state_dict
# needs an identically-shaped model to load INTO, so the definition has to live
# somewhere reusable.
import torch
import torch.nn as nn

def makeModel():

  model = nn.Sequential(
    nn.Linear(1,10),
    nn.ReLU(),
    nn.Linear(10,1),
  )

  return model

In [22]:
# state_dict() is an OrderedDict of {parameter name: tensor}. That is the
# entire learned content of a model - everything else is code.
model = makeModel()
model.state_dict()

OrderedDict([('0.weight',
              tensor([[ 0.4583],
                      [-0.0610],
                      [-0.0633],
                      [ 0.8672],
                      [ 0.5876],
                      [-0.9835],
                      [ 0.0924],
                      [-0.0370],
                      [-0.4772],
                      [ 0.1016]])),
             ('0.bias',
              tensor([ 0.4347, -0.0213,  0.8268,  0.0751,  0.1294, -0.9751, -0.5281,  0.4955,
                      -0.2085, -0.4473])),
             ('2.weight',
              tensor([[ 0.0304, -0.0159,  0.1917, -0.1851, -0.1612, -0.2996,  0.0431, -0.0770,
                       -0.0970, -0.1210]])),
             ('2.bias', tensor([-0.2588]))])

In [23]:
# Save just the weights. Convention is a .pt or .pth extension.
# Saving Model
torch.save(model.state_dict(),"TestModel.pt")

In [24]:
# Load: build the architecture first, then pour the weights in.
# A shape mismatch raises here, which is the point - it fails loudly instead of
# silently producing a wrong model.
# In production add model.eval() after this, and map_location='cpu' when the
# weights were trained on a GPU.
# Loading Model
LodedModel = makeModel()

LodedModel.load_state_dict(torch.load('TestModel.pt'))

<All keys matched successfully>

In [29]:
# Same input, same output from both models: the weights transferred exactly.
print(model(torch.tensor(1,dtype=torch.float).view(1,-1)))
print(LodedModel(torch.tensor(1,dtype=torch.float).view(1,-1)))

print("The outputs are the same, so the model weight are the same ones")

tensor([[-0.4105]], grad_fn=<AddmmBackward0>)
tensor([[-0.4105]], grad_fn=<AddmmBackward0>)
The outputs are the same, so the model weight are the same ones
